# RAG Pipeline — Health Plan Policy Documents (2026)

Semantic-retrieval RAG over 5 policy PDFs using **Azure OpenAI** + **ChromaDB**.
1. Load & extract text from PDFs
2. Chunk by section
3. Embed (`text-embedding-3-small`) & store in a Chroma collection
4. Semantic retrieval
5. Generate grounded, cited answers (`gpt-4.1`)

**Setup:** put a `.env` next to this notebook:
```
AZURE_OPENAI_ENDPOINT="https://....openai.azure.com/"
AZURE_OPENAI_API_KEY="..."
AZURE_OPENAI_MODEL="gpt-4.1"
AZURE_OPENAI_EMBEDDING_MODEL="text-embedding-3-small"
AZURE_OPENAI_MODEL_SECONDARY="gpt-5.1"
```
> `AZURE_OPENAI_MODEL*` values are the **deployment names** in your Azure resource.

## 1. Install dependencies
Your base env already has `chromadb`, `openai`, `pypdf`, `python-dotenv`.

In [ ]:
%pip install -q chromadb openai pypdf python-dotenv

## 2. Imports & configuration

In [ ]:
import os
import re
import glob

import chromadb
from pypdf import PdfReader
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
GEN_MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1")
EMBED_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")

assert ENDPOINT and API_KEY, "Azure endpoint/key not found. Check your .env file."

# --- Config ---
PDF_DIR = r"C:\Users\KA48795\Documents\HPP AI Learning Sessions\zs_ai_participants-main\notebook\Phase_2\data\healthcare_policies"  # folder holding the 5 PDFs
COLLECTION = "policies_2026"
TOP_K = 4

client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=API_KEY, api_version=API_VERSION)
print("Azure OpenAI client ready.")

## 3. Load & extract text from PDFs

In [ ]:
def load_pdfs(pdf_dir):
    docs = []
    paths = sorted(glob.glob(os.path.join(pdf_dir, "*.pdf")))
    if not paths:
        raise FileNotFoundError(f"No PDFs found in {pdf_dir!r}. Place the 5 files there.")
    for p in paths:
        reader = PdfReader(p)
        text = "\n".join((page.extract_text() or "") for page in reader.pages)
        docs.append({"source": os.path.basename(p), "text": text})
    return docs

docs = load_pdfs(PDF_DIR)
print(f"Loaded {len(docs)} documents:")
for d in docs:
    print(f"  - {d['source']} ({len(d['text'])} chars)")

## 4. Chunk documents by section

In [ ]:
SECTION_RE = re.compile(r"(SECTION\s+\d+:[^\n]*)", re.IGNORECASE)

def chunk_by_section(doc, max_chars=1200):
    parts = SECTION_RE.split(doc["text"])
    chunks = []
    preamble = parts[0].strip()
    if preamble:
        chunks.append({"source": doc["source"], "section": "Header/Metadata", "text": preamble})
    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        content = f"{header}\n{body}".strip()
        for j in range(0, len(content), max_chars):
            chunks.append({"source": doc["source"], "section": header, "text": content[j:j + max_chars]})
    return chunks

all_chunks = []
for d in docs:
    all_chunks.extend(chunk_by_section(d))
print(f"Total chunks: {len(all_chunks)}")

## 5. Embed & store in ChromaDB
We compute embeddings with Azure and pass them to Chroma directly (Chroma stores the vectors + metadata and handles cosine search). Using an in-memory client here; swap to `PersistentClient(path=...)` to persist.

In [ ]:
def embed_texts(texts, batch_size=64):
    vecs = []
    for i in range(0, len(texts), batch_size):
        resp = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch_size])
        vecs.extend(d.embedding for d in resp.data)
    return vecs

chroma_client = chromadb.Client()  # in-memory
try:
    chroma_client.delete_collection(COLLECTION)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION, metadata={"hnsw:space": "cosine"})

texts = [c["text"] for c in all_chunks]
embeddings = embed_texts(texts)
collection.add(
    ids=[f"chunk_{i}" for i in range(len(all_chunks))],
    embeddings=embeddings,
    documents=texts,
    metadatas=[{"source": c["source"], "section": c["section"]} for c in all_chunks],
)
print(f"Stored {collection.count()} chunks in Chroma collection '{COLLECTION}'.")

## 6. Semantic retrieval

In [ ]:
def retrieve(query, k=TOP_K):
    q_emb = embed_texts([query])
    res = collection.query(query_embeddings=q_emb, n_results=k)
    out = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        out.append({"text": doc, "source": meta["source"], "section": meta["section"],
                    "score": 1 - dist})  # cosine distance -> similarity
    return out

for r in retrieve("How many physical therapy visits before prior authorization on Gold PPO?"):
    print(f"[{r['score']:.3f}] {r['source']} | {r['section']}")

## 7. Generation — grounded answer with citations

In [ ]:
SYSTEM_PROMPT = (
    "You are a health-plan policy assistant. Answer ONLY using the provided context. "
    "If the answer is not in the context, say you don't have that information. "
    "Cite the source document and section for each fact you use."
)

def build_context(results):
    return "\n\n".join(
        f"[{i}] Source: {r['source']} | {r['section']}\n{r['text']}"
        for i, r in enumerate(results, 1)
    )

def rag_answer(query, k=TOP_K):
    results = retrieve(query, k)
    resp = client.chat.completions.create(
        model=GEN_MODEL,
        max_tokens=1000,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Context:\n{build_context(results)}\n\nQuestion: {query}"},
        ],
    )
    return resp.choices[0].message.content, results

answer, sources = rag_answer(
    "How many physical therapy visits are allowed before prior authorization on Gold PPO vs Silver HMO?"
)
print(answer)
print("\n--- Retrieved from ---")
for s in sources:
    print(f"  [{s['score']:.3f}] {s['source']} | {s['section']}")

## 8. Try your own questions

In [ ]:
questions = [
    "Does emergency MRI require prior authorization?",
    "What is the chiropractic visit limit on each plan?",
    "How long does a provider have to file an appeal after a denial?",
    "Do Gold PPO members need a referral to see a specialist?",
]

for q in questions:
    ans, _ = rag_answer(q)
    print(f"Q: {q}\nA: {ans}\n{'='*80}")